> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `compat`, and
the header cell below rewrites `compat.py`; importing the package first means no later import
reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none | none: `compat` never touches the bus |
| **Result** | not run yet on the hardware | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): on CPython the timing functions go through a swappable clock object,
so `micropython_i2c.sim.clock.SimClock` can run the drivers and fakes in simulated time. On
MicroPython they stay the `time` module's own functions: the board pays nothing for it, and
`clock_set` refuses.

# Python file
location: `src/micropython_i2c/compat.py`

tests: `tests/test_compat.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/compat.py
"""Small shims so the package imports and runs under both MicroPython and CPython.

Board code uses MicroPython's ``time.sleep_ms`` / ``ticks_*`` family and ``_thread`` locks.
CPython has neither the ``ticks`` API nor ``sleep_ms``. Everything in the package imports
timing from here instead of from ``time`` directly.

On CPython the timing functions go through a clock object that ``clock_set`` can swap. The
default, ``RealClock``, is the wall clock; ``micropython_i2c.sim.clock.SimClock`` is
simulated time, for running drivers against fake parts without waiting. On MicroPython the
functions are the ``time`` module's own and ``clock_set`` raises ``NotImplementedError``.

Generated by ``dev_notebooks/compat_devNB``: edit the notebook, not this file.
"""

# Imports
## External
`time` only: it is the one module both MicroPython and CPython have with the same name.

## Internal
None: everything else in the package imports from here.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
import time

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_compat.py
import pytest

from micropython_i2c.compat import (
    NullLock,
    RealClock,
    clock_get,
    clock_set,
    default_lock,
    sleep_ms,
    sleep_us,
    ticks_add,
    ticks_diff,
    ticks_ms,
    ticks_us,
)

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

# Intent
**Purpose:** one place for the things MicroPython and CPython do differently: millisecond and
microsecond sleeps, the wrapping `ticks` counters, and locks.

**Why it exists:** the package runs on the board and imports on the host. Without this module
every driver would need `try: time.sleep_ms ... except AttributeError` of its own, and host
tests would need monkeypatching.

**The clock.** On CPython every timing call goes through one clock object:

```mermaid
flowchart LR
    D[driver / fake] -->|sleep_us, ticks_ms| C[compat]
    C -->|MicroPython| T[time module]
    C -->|CPython| K{clock}
    K --> R[RealClock: wall clock]
    K --> S[SimClock: simulated time]
```

`clock_set(clock)` swaps it and returns the previous one. A clock needs two methods:
`ticks_us()` (microseconds since the clock started) and `sleep_us(us)`.

**Used by:** every driver (waits), the fakes (measurement timing), `sim.clock`.

**Not its job:** wrapping arithmetic on CPython. CPython ticks never wrap, so `ticks_add` and
`ticks_diff` are plain `+` and `-` there.

## What the module defines

| Name | What it is |
|---|---|
| `sleep_ms`, `sleep_us` | sleep, through the clock on CPython |
| `ticks_ms`, `ticks_us` | monotonic counters, through the clock on CPython |
| `ticks_add`, `ticks_diff` | tick arithmetic (wrapping on MicroPython) |
| `RealClock` | the CPython wall clock, the default |
| `clock_get`, `clock_set` | read and swap the CPython clock |
| `NullLock` | a lock that does nothing, for builds without `_thread` |
| `default_lock` | a real lock where there is one, else a `NullLock` |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
__all__ = [
    "NullLock",
    "RealClock",
    "clock_get",
    "clock_set",
    "default_lock",
    "sleep_ms",
    "sleep_us",
    "ticks_add",
    "ticks_diff",
    "ticks_ms",
    "ticks_us",
]

## `RealClock`
The CPython wall clock. `ticks_us` counts from when the clock was made, so simulated and real
time both start near zero and a signal written as "ramp from t = 0 s to t = 60 s" means the
same on either. MicroPython never uses it, but the class exists there too so imports match.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
class RealClock:
    """Wall-clock time for CPython: ``time.monotonic`` and ``time.sleep``.

    Attributes:
        t0: ``time.monotonic()`` when the clock was made; ``ticks_us`` counts from it.
    """

    def __init__(self):
        self.t0 = time.monotonic() if hasattr(time, "monotonic") else 0

    def ticks_us(self):
        """Microseconds since the clock was made."""
        return int((time.monotonic() - self.t0) * 1_000_000)

    def sleep_us(self, us):
        """Sleep ``us`` microseconds of real time."""
        if us > 0:
            time.sleep(us / 1_000_000)

### validation/test

#### validation: nominal usage
A fresh clock starts near zero and moves forward with real time.

In [ ]:
rc = RealClock()
a = rc.ticks_us()
rc.sleep_us(2000)
b = rc.ticks_us()
print("start:", a, "us | after a 2 ms sleep:", b, "us | elapsed >= 2000:", b - a >= 2000)

#### validation: anti-nominal usage
None: `RealClock` has no failure path. A zero or negative sleep returns at once.

In [ ]:
rc.sleep_us(-5)
print("negative sleep returned at once")

#### unit

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_real_clock_starts_near_zero_and_advances():
    rc = RealClock()
    a = rc.ticks_us()
    rc.sleep_us(1000)
    b = rc.ticks_us()
    assert 0 <= a < 1_000_000
    assert b - a >= 1000

## The timing functions
On MicroPython these are the `time` module's own functions, bound once at import. On CPython
each one calls the current clock, so swapping the clock changes every caller at once, even
modules that imported `sleep_us` by name before the swap.

`ticks_add` and `ticks_diff` wrap on MicroPython (the counters roll over); on CPython they are
plain arithmetic, because the counters never wrap there.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
try:
    sleep_ms = time.sleep_ms
    sleep_us = time.sleep_us
    ticks_ms = time.ticks_ms
    ticks_us = time.ticks_us
    ticks_add = time.ticks_add
    ticks_diff = time.ticks_diff
    _MICROPYTHON = True
except AttributeError:  # CPython
    _MICROPYTHON = False
    _clock = RealClock()

    def sleep_ms(ms):
        """Sleep ``ms`` milliseconds on the current clock."""
        _clock.sleep_us(int(ms * 1000))

    def sleep_us(us):
        """Sleep ``us`` microseconds on the current clock."""
        _clock.sleep_us(int(us))

    def ticks_ms():
        """Milliseconds on the current clock (does not wrap on CPython)."""
        return _clock.ticks_us() // 1000

    def ticks_us():
        """Microseconds on the current clock (does not wrap on CPython)."""
        return _clock.ticks_us()

    def ticks_add(ticks, delta):
        """Offset a tick value; plain addition since CPython ticks never wrap."""
        return ticks + delta

    def ticks_diff(a, b):
        """Signed difference ``a - b`` between two tick values."""
        return a - b

## `clock_get` / `clock_set`
Read and swap the CPython clock. `clock_set` returns the previous clock so a caller can put it
back (`sim.clock.SimClock` does this as a context manager). On MicroPython there is nothing to
swap: the timing functions are the hardware's.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
def clock_get():
    """The clock the timing functions use (CPython only).

    Raises:
        NotImplementedError: On MicroPython, where the timing functions are ``time``'s own.
    """
    if _MICROPYTHON:
        raise NotImplementedError("the clock is fixed on MicroPython")
    return _clock


def clock_set(clock):
    """Make every timing function use ``clock``; return the previous clock (CPython only).

    Args:
        clock: Any object with ``ticks_us()`` and ``sleep_us(us)``.

    Raises:
        NotImplementedError: On MicroPython.
        TypeError: If ``clock`` lacks ``ticks_us`` or ``sleep_us``.
    """
    global _clock
    if _MICROPYTHON:
        raise NotImplementedError("the clock is fixed on MicroPython")
    for name in ("ticks_us", "sleep_us"):
        if not callable(getattr(clock, name, None)):
            raise TypeError("clock {!r} has no {}() method".format(clock, name))
    previous, _clock = _clock, clock
    return previous

### validation/test
A demo clock for the cells below: time moves only when something sleeps, like `SimClock`
(which lives in `sim.clock` and is validated there).

In [ ]:
%%extract ../tests/test_compat.py -a
class StepClock:
    """Simulated time for the tests: moves only on sleep."""

    def __init__(self):
        self.now = 0

    def ticks_us(self):
        return self.now

    def sleep_us(self, us):
        self.now += max(0, int(us))

#### validation: nominal usage
Swap in the step clock: a 10-second sleep returns at once and the counters move by exactly 10
seconds. Then put the real clock back.

In [ ]:
import time as _t

sc = StepClock()
before = clock_set(sc)
try:
    wall = _t.monotonic()
    sleep_ms(10_000)
    print("ticks_ms:", ticks_ms(), "| ticks_us:", ticks_us(), "| wall time spent: {:.3f} s".format(_t.monotonic() - wall))
finally:
    clock_set(before)
print("restored the previous clock:", clock_get() is before)

#### validation: anti-nominal usage

In [ ]:
try:
    clock_set(object())  # a clock needs ticks_us() and sleep_us()
except TypeError as err:
    print("raised TypeError, as expected:\n    {}".format(err))
else:
    raise AssertionError("clock_set(object()) should have raised TypeError")
finally:
    print("clock unchanged:", clock_get() is before)

#### unit

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_swapped_clock_drives_every_timing_function():
    sc = StepClock()
    before = clock_set(sc)
    try:
        sleep_ms(5)
        sleep_us(250)
        assert ticks_us() == 5250
        assert ticks_ms() == 5
    finally:
        clock_set(before)

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_clock_set_returns_the_previous_clock():
    sc = StepClock()
    before = clock_set(sc)
    try:
        assert clock_get() is sc
        assert before is not sc
    finally:
        assert clock_set(before) is sc
    assert clock_get() is before

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_clock_set_refuses_an_object_that_is_not_a_clock():
    before = clock_get()
    with pytest.raises(TypeError, match="ticks_us"):
        clock_set(object())
    assert clock_get() is before

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_tick_arithmetic():
    assert ticks_add(100, 25) == 125
    assert ticks_diff(100, 125) == -25
    assert ticks_diff(ticks_add(7, 3), 7) == 3

## `NullLock`
A lock that does nothing, for MicroPython builds without `_thread`. It implements just what
the call sites use (`acquire`, `release`, `with`), so they need no conditional logic. It is
correct whenever there is no concurrency to guard against.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
try:
    import _thread
except ImportError:  # MicroPython build without threading
    _thread = None


class NullLock:
    """No-op stand-in for a mutex on single-threaded builds."""

    def acquire(self, *args):
        return True

    def release(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False

## `default_lock`
The lock a bus gets: a real `_thread` lock where the build has threads, else a `NullLock`.
Every connection and mux on one bus shares one lock, so callers normally get it from the
manager.

In [ ]:
%%extract ../src/micropython_i2c/compat.py -a
def default_lock():
    """Return a real ``_thread`` lock where available, else a ``NullLock``."""
    return _thread.allocate_lock() if _thread is not None else NullLock()

### validation/test

#### validation: nominal usage
CPython has `_thread`, so the default lock is a real one: a second non-blocking acquire fails
while it is held.

In [ ]:
lk = default_lock()
with lk:
    print("real lock:", not isinstance(lk, NullLock), "| second acquire while held:", lk.acquire(False))
nl = NullLock()
with nl:
    print("NullLock acquire inside with:", nl.acquire())

#### validation: anti-nominal usage
None: neither lock has a failure path the package relies on.

#### unit

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_default_lock_is_exclusive_on_cpython():
    lk = default_lock()
    with lk:
        assert lk.acquire(False) is False
    assert lk.acquire(False) is True
    lk.release()

In [ ]:
%%extract ../tests/test_compat.py -a --strip-ipytest
%%ipytest


def test_null_lock_never_blocks():
    nl = NullLock()
    with nl as same:
        assert same is nl
        assert nl.acquire() is True
    nl.release()

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests** (one per unit test above that makes sense
on MicroPython, same name), then the tally and disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.compat import (
    NullLock,
    clock_set,
    default_lock,
    sleep_ms,
    sleep_us,
    ticks_add,
    ticks_diff,
    ticks_ms,
    ticks_us,
)
from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)

check_reset()

## Matching tests

**Matches** `test_tick_arithmetic`, plus the wrap the host cannot show: on MicroPython
`ticks_diff` stays correct across the counter's roll-over.

**What to do:** nothing.

In [ ]:
check("test_tick_arithmetic",
      ticks_add(100, 25) == 125 and ticks_diff(100, 125) == -25
      and ticks_diff(ticks_add(7, 3), 7) == 3)  # fmt: skip
near_top = ticks_add(0, -5)  # 5 ticks before the counter wraps
check("ticks_diff across the wrap", ticks_diff(ticks_add(near_top, 10), near_top) == 10)

**Matches** `test_swapped_clock_drives_every_timing_function`, the board's half: the timing
functions are the hardware's, and a 5 ms sleep takes 5 ms.

**What to do:** nothing.

In [ ]:
t0 = ticks_us()
sleep_ms(5)
dt = ticks_diff(ticks_us(), t0)
check("sleep_ms(5) takes 5 ms", 5000 <= dt < 7000, "{} us".format(dt))
t0 = ticks_ms()
sleep_us(2000)
check("ticks_ms and sleep_us agree", 2 <= ticks_diff(ticks_ms(), t0) <= 3)

**Matches** `test_clock_set_refuses_an_object_that_is_not_a_clock`: on the board every clock
is refused, because the timing functions are fixed.

**What to do:** nothing.

In [ ]:
check_raises("clock_set refuses on MicroPython", NotImplementedError, clock_set, object())

**Matches** `test_default_lock_is_exclusive_on_cpython` and `test_null_lock_never_blocks`.

**What to do:** nothing.

In [ ]:
lk = default_lock()
if isinstance(lk, NullLock):
    check("test_null_lock_never_blocks (no _thread on this build)", lk.acquire())
else:
    lk.acquire()
    held = lk.acquire(0)
    lk.release()
    check("test_default_lock_is_exclusive (_thread lock)", held is False)
nl = NullLock()
with nl:
    check("test_null_lock_never_blocks", nl.acquire() is True)

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))